In [ ]:
# for combing MT Binding energies into one finding common
import pandas as pd
import re

# ─── USER SETTINGS ────────────────────────────────────────────────────────────
files   = {
    "DYRK1A": "DYRK1A_Affinity_result_file.csv",
    "ABL1":   "ABL1_Affinity_result_file.csv",
    "TTBK1":  "TTBK1_Affinity_result_file.csv"
}
ligand_col    = "Name of the ligand"
energy_cutoff = -8.5   # keep ligands with ΔG ≤ this (more negative = stronger)
# ────────────────────────────────────────────────────────────────────────────────

# 1) Read & filter each file, renaming its energy column to e.g. ΔG_DYRK1A
filtered = {}
for target, fname in files.items():
    df = pd.read_csv(fname)
    # detect the “Binding Free Energy” column
    energy_cols = [c for c in df.columns if re.search(r"Binding Free Energy", c, re.I)]
    if not energy_cols:
        raise ValueError(f"No Binding Free Energy column in {fname}")
    col = energy_cols[0]
    # coerce to numeric & drop bad rows
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=[col])
    # filter
    df_filt = df[df[col] <= energy_cutoff].copy()
    # rename for clarity
    df_filt = df_filt.rename(columns={col: f"ΔG_{target}"})
    filtered[target] = df_filt

# 2) Find common ligand names across all three filtered sets
sets = [set(df[ligand_col]) for df in filtered.values()]
common = set.intersection(*sets)
print(f"Found {len(common)} ligands passing ΔG ≤ {energy_cutoff} in all three targets.")

# 3) Build a combined DataFrame by inner‐joining on the ligand name
#    Start from the first target, then merge the others
targets = list(files.keys())
combined = filtered[targets[0]][[ligand_col, f"ΔG_{targets[0]}"]]
for target in targets[1:]:
    combined = combined.merge(
        filtered[target][[ligand_col, f"ΔG_{target}"]],
        on=ligand_col,
        how="inner"
    )

# 4) Optionally restrict to exactly the common set (should already be the case)
combined = combined[combined[ligand_col].isin(common)].reset_index(drop=True)

# 5) Save outputs
combined.to_csv("common_compounds_with_energies.csv", index=False)
with open("common_compound_list.txt", "w") as fo:
    for lig in combined[ligand_col]:
        fo.write(f"{lig}\n")

print("▶ Wrote detailed table: common_compounds_with_energies.csv")
print("▶ Wrote ligand list:     common_compound_list.txt")


Found 353 ligands passing ΔG ≤ -8.5 in all three targets.
▶ Wrote detailed table: common_compounds_with_energies.csv
▶ Wrote ligand list:     common_compound_list.txt


In [ ]:
#for fetching mettadata from parent file
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT')
import pandas as pd

# Load your filtered data
final_df = pd.read_csv('common_compounds_with_energies.csv')

# Load the coconut data
coconut_df = pd.read_csv('coconut_csv-06-2025.csv')

# Preview to identify the correct identifier column names
print(final_df.columns)
print(coconut_df.columns)

# Assume the identifier in coconut_df is 'coconut_ID' (replace with actual column if different)
# Merge on the identifier, keeping all rows from final_df
merged_df = final_df.merge(
    coconut_df[['Identifier', 'canonical_smiles', 'name','iupac_name']], 
    left_on='Name of the ligand (fixed)', 
    right_on='Identifier', 
    how='left'
)


# Save the merged file
merged_df.to_csv('new_combined_annotated.csv', index=False)


/tmp/ipykernel_756119/3775409694.py:9: DtypeWarning: Columns (6,7,8,9,10,12,13,14,15,16,17,18,19,20,21,23,24,25,26,27,28,30,38,39,41,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,66,68) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut_df = pd.read_csv('coconut_csv-06-2025.csv')


Index(['Name of the ligand', 'Name of the ligand (fixed)', 'ΔG_DYRK1A',
       'ΔG_ABL1', 'ΔG_TTBK1'],
      dtype='object')
Index(['Identifier', 'canonical_smiles', 'standard_inchi',
       'standard_inchi_key', 'name', 'iupac_name', 'annotation_level',
       'total_atom_count', 'heavy_atom_count', 'molecular_weight',
       'exact_molecular_weight', 'molecular_formula', 'alogp',
       'topological_polar_surface_area', 'rotatable_bond_count',
       'hydrogen_bond_acceptors', 'hydrogen_bond_donors',
       'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski',
       'lipinski_rule_of_five_violations', 'aromatic_rings_count',
       'qed_drug_likeliness', 'formal_charge', 'fractioncsp3',
       'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar',
       'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework',
       'np_likeness', 'chemical_class', 'chemical_sub_class',
       'chemical_super_class', 'direct_parent_classification',
       '

In [8]:
#FINDING COMMON COMPOUNDS IN TWO FILES
import pandas as pd

# ─── CONFIG ───────────────────────────────────────────────────────────────────
file_a     = "SwissADME Final.csv"
file_b     = "After_admet_processed.csv"
column     = "Canonical SMILES"   # change this to the exact column header
output_txt = "common_values.txt"
output_csv = "common_after_ADME.csv"
# ────────────────────────────────────────────────────────────────────────────────

# 1) Read both CSVs
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# 2) Ensure the column exists
for df, fn in [(df_a, file_a), (df_b, file_b)]:
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in {fn}")

# 3) Extract sets of unique values
set_a = set(df_a[column].dropna().astype(str).unique())
set_b = set(df_b[column].dropna().astype(str).unique())

# 4) Compute intersection
common = sorted(set_a & set_b)
print(f"Found {len(common)} common values in column '{column}'.")

# # 5) Save to text file
# with open(output_txt, "w") as fo:
#     for val in common:
#         fo.write(val + "\n")
# print(f"▶ Common values written to {output_txt}")

# # 6) (Optional) Save to CSV
pd.DataFrame({column: common}).to_csv(output_csv, index=False)
# print(f"▶ Common values written to {output_csv}")


Found 25 common values in column 'Canonical SMILES'.


In [4]:
## IMPORTANT 
# getting all 3d sdfs (multitargeted) from coconut db from 3d_zip.sdf file 
from rdkit import Chem
import pandas as pd
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/final MT')
def has_all_zero_z(mol):
    """Check if all Z coordinates in the first conformer are zero."""
    conf = mol.GetConformer()
    zs = [conf.GetAtomPosition(i).z for i in range(mol.GetNumAtoms())]
    return all(abs(z) < 1e-4 for z in zs)  # tolerance for floating point

# 1. Load your list of identifiers from CSV/table
ids_df = pd.read_csv("After_admet_processed.csv")  # Update with your file name
wanted_ids = set(ids_df['Identifier'].astype(str))  # Ensure all IDs are strings

# 2. Prepare output directories
output_dir = "before gnina 3D"
zero_z_dir = "zero_before_gnina"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(zero_z_dir, exist_ok=True)

# 3. Iterate through the COCONUT 3D SDF and extract matching molecules
supplier = Chem.SDMolSupplier("coconut_sdf_3d-06-2025.sdf")
found = 0
zero_z_found = 0

for mol in supplier:
    if mol is None:
        continue
    ident = mol.GetProp("IDENTIFIER") if mol.HasProp("IDENTIFIER") else None
    if ident in wanted_ids:
        if has_all_zero_z(mol):
            writer = Chem.SDWriter(os.path.join(zero_z_dir, f"{ident}.sdf"))
            zero_z_found += 1
        else:
            writer = Chem.SDWriter(os.path.join(output_dir, f"{ident}.sdf"))
            found += 1
            if found % 100 == 0:
                print(f"{found} molecules with valid 3D written...")
        writer.write(mol)
        writer.close()

print(f" Extraction complete. {found} valid 3D SDF files written to '{output_dir}/'.")
print(f"  {zero_z_found} molecules with all-zero Z coordinates written to '{zero_z_dir}/'.")


[11:40:46] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:47] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:48] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:49] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:49] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:50] Warning: molecule is tagged as 3D, but all Z coords are zero
[11:40:50] WARNING: not removing hydrogen atom with neighbor tha

 Extraction complete. 25 valid 3D SDF files written to 'before gnina 3D/'.
  0 molecules with all-zero Z coordinates written to 'zero_before_gnina/'.


In [5]:
import rdkit
print(rdkit.__version__)


2022.09.5
